# 01 — ZIT only  — PP+HP 버전 (스태킹 다양성용)

기존 `01_zit_only.ipynb`(HP만 흔드는 노트북)의 **쌍둥이** — 여기에 **PP 6축을 Optuna trial 축에 추가**해서 같은 모델이라도 다른 전처리에서 학습한 OOF를 만든다 → 스태킹 base 다양성 ↑.

- PP 6축: `missing_threshold / corr_threshold / add_indicator / indicator_threshold / spatial_max_dist / post_impute_corr_threshold` ([pp_hp_strategy.md §3](../../pp_hp_strategy.md)). 나머지는 PP_FIXED와 동일하게 고정. trial마다 `pp_hpo.make_cached_preprocess`로 전처리(LRU 캐시).
- HP·anchor·CV·후처리·산출물은 `01_zit_only.ipynb`와 동일. anchor에 PP_FIXED 값을 `pp_*` 키로 추가(corr 0.90->0.88).
- 출력 폴더만 다름: `4_output/.../pphp/`.

> ⚠ pp+hp는 trial마다 전처리(spatial impute 포함)를 다시 도므로 hp-only보다 느리다. `PP_CACHE_SIZE`로 캐시 보관 개수 조정.


## 1. 환경 설정 + import

Colab/Local 자동 감지. Colab 사용 시 `GDRIVE_MODELING_ID` 채울 것 (modeling.zip 공유 ID — 신규 `3_modeling/`을 zip으로 업로드 후 ID 입력).

In [ ]:
import os, sys

# Google Drive 파일 ID들 — Colab에서 코드/데이터/모듈 zip을 자동으로 받아 풀 때 사용 (로컬은 무시)
GDRIVE_CODE_ID         = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'   # code.zip = setup.py + utils/
GDRIVE_DATASET_ID      = '1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO'   # dataset.zip = 원본 CSV 4개
GDRIVE_PREPROCESSING_ID = '1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr'  # preprocessing.zip = cleaning/outlier/scaling/meta_features 등
GDRIVE_MODELING_ID     = '1Vrn5LBl611rWbag7d09LZH68_lfpu6wP'   # modeling.zip = 3_modeling/modules (코드 수정 시 재업로드 필요)
GDRIVE_OUTPUT_ID       = '1ts73qEMmjX8cKIb-QeDQ-TMeyudFGWzs'   # 4_output.zip = 기존 실험 산출물 (RESUME 시 복원용)
RESUME                 = True   # True=기존 optuna db에 trial을 이어 붙임 / False=처음부터 (이미 db가 있으면 의도적으로 에러)

# Colab이면 필요한 zip들을 받아 풀고(이미 풀려 있으면 skip), 로컬이면 ../../setup.py만 실행
try:
    import google.colab
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system(f'gdown {GDRIVE_DATASET_ID} -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system(f'gdown {GDRIVE_PREPROCESSING_ID} -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if GDRIVE_MODELING_ID and not os.path.exists('/content/project/3_modeling/modules/zit.py'):
        os.system(f'gdown {GDRIVE_MODELING_ID} -O /content/modeling.zip')
        os.makedirs('/content/project/3_modeling', exist_ok=True)
        os.system('unzip -qo /content/modeling.zip -d /content/project/3_modeling')
    # RESUME이면 이전 4_output을 통째로 복원해 둠 (이미 이 실험 폴더가 있으면 skip) → 아래 create_study가 이어서 받음
    if RESUME and GDRIVE_OUTPUT_ID and not os.path.exists('/content/project/4_output/01_zit'):
        os.system(f'gdown {GDRIVE_OUTPUT_ID} -O /content/4_output.zip')
        os.system('unzip -qo /content/4_output.zip -d /content/project')
        os.remove('/content/4_output.zip')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

# 전처리 모듈(2_preprocessing)을 패키지 접두사 없이 import 하려고 경로에 추가
PP_DIR = os.path.join(PROJECT_ROOT, '2_preprocessing')
if PP_DIR not in sys.path:
    sys.path.insert(0, PP_DIR)

# `from modules import ...` 가 3_modeling/modules를 찾게
MOD_DIR = os.path.join(PROJECT_ROOT, '3_modeling')
if MOD_DIR not in sys.path:
    sys.path.insert(0, MOD_DIR)

from modules import preprocess, hpo, postprocess, pp_hpo
from modules.zit import ZITboostRegressor                # ZI-Tweedie + LightGBM EM 회귀 모델
from meta_features import add_meta_features               # die_xy / position 같은 메타피처 추가 헬퍼

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from sklearn.model_selection import KFold

import logging, time
logging.getLogger('lightgbm').setLevel(logging.ERROR)   # LGBM 로그 억제
optuna.logging.set_verbosity(optuna.logging.WARNING)    # Optuna 로그 억제

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'optuna v{optuna.__version__}')

## 2. 실험 설정

- `N_JOBS = 7` (학원 14코어 / 노트북 2개 병렬, strategy_common §8)
- `PP_FIXED` (strategy_common §1)
- `ZIT_ONLY_ANCHOR` (strategy.md §4 — trial 99 best)
- `ZIT_SEARCH` (strategy.md §6 표 그대로)

In [ ]:
# 실험 식별 — 출력 폴더/DB 파일명에 들어감
EXP_ID = 'zit-only-pphp'
USER   = 'jh'

# Optuna 예산
N_TRIALS         = 500
TIMEOUT_SEC      = 90 * 60 * 60  # 초 단위, None=무제한 (Colab 세션 타임아웃 대비)
N_FOLDS          = 5
N_JOBS           = 7   # 모델 학습 병렬도 (가용 코어 수에 맞춰)
N_STARTUP_TRIALS = 30  # TPE가 학습을 시작하기 전 무작위 trial 수

# 출력 경로 — EXP_ID 끝자리('002')를 하위 폴더명으로
OUT_DIR = os.path.join(OUTPUT_DIR, '01_zit', 'zit_only', EXP_ID.split('-')[-1])
os.makedirs(OUT_DIR, exist_ok=True)
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')   # study가 여기에 자동 저장 (RESUME 시 여기서 이어서)

CLIP_Y_EXTREME = True   # train y의 max(=1.0, 단 1건)를 두 번째로 큰 값으로 clip — 모델 학습 입력에만 적용

PP_CACHE_SIZE = 2   # cached_preprocess가 보관할 PP 조합 개수 (cleaned 3-split ≈ 1~1.5GB/개) — 메모리 보고 조정
# PP는 더 이상 고정값이 아니라 Optuna 탐색 6축 (pp_hpo.PP_SEARCH_CANDIDATES — pp_hp_strategy.md §3)

# anchor — 1차 실험에서 찾은 best HP (이걸 study의 trial 0으로 강제 enqueue → 1차 성능을 절대 잃지 않음)
ZIT_ONLY_ANCHOR = {
    'zeta':                  1.149,
    'n_em_iters':            13,
    'mu_n_estimators':       240,
    'mu_learning_rate':      0.00309,
    'mu_num_leaves':         212,
    'mu_max_depth':          3,
    'mu_min_child_samples':  132,
    'mu_subsample':          0.649,
    'mu_colsample_bytree':   0.255,
    'mu_reg_alpha':          0.00576,
    'mu_reg_lambda':         0.00155,
    'pi_n_estimators':       125,
    'pi_learning_rate':      0.0408,
    'pi_num_leaves':         165,
    'pi_max_depth':          11,
    'pi_min_child_samples':  38,
    'phi_n_estimators':      57,
    'phi_learning_rate':     0.00628,
    'phi_num_leaves':        65,
    'phi_max_depth':         4,
    'phi_min_child_samples': 190,
}
ANCHOR_TAU_PI = 0.944   # anchor의 τ_π (π가 이 값 초과인 die는 예측을 0으로)

# 탐색 공간 — anchor 주변 ±30%로 좁힌 범위 ({key: {type, low, high, log}} 형태, hpo.sample_from_space가 풂)
ZIT_SEARCH = {
    'zeta':                  {'type': 'float', 'low': 1.05,   'high': 1.50,  'log': False},
    'n_em_iters':            {'type': 'int',   'low': 10,     'high': 20},
    'mu_n_estimators':       {'type': 'int',   'low': 170,    'high': 320},
    'mu_learning_rate':      {'type': 'float', 'low': 0.0021, 'high': 0.0046, 'log': True},
    'mu_num_leaves':         {'type': 'int',   'low': 148,    'high': 280},
    'mu_max_depth':          {'type': 'int',   'low': 3,      'high': 5},
    'mu_min_child_samples':  {'type': 'int',   'low': 90,     'high': 240},
    'mu_subsample':          {'type': 'float', 'low': 0.50,   'high': 0.85,  'log': False},
    'mu_colsample_bytree':   {'type': 'float', 'low': 0.18,   'high': 0.35,  'log': False},
    'mu_reg_alpha':          {'type': 'float', 'low': 0.002,  'high': 0.015, 'log': True},
    'mu_reg_lambda':         {'type': 'float', 'low': 5e-4,   'high': 5e-3,  'log': True},
    'pi_n_estimators':       {'type': 'int',   'low': 90,     'high': 240},
    'pi_learning_rate':      {'type': 'float', 'low': 0.028,  'high': 0.060, 'log': True},
    'pi_num_leaves':         {'type': 'int',   'low': 115,    'high': 220},
    'pi_max_depth':          {'type': 'int',   'low': 8,      'high': 16},
    'pi_min_child_samples':  {'type': 'int',   'low': 25,     'high': 55},
    'phi_n_estimators':      {'type': 'int',   'low': 40,     'high': 80},
    'phi_learning_rate':     {'type': 'float', 'low': 0.004,  'high': 0.010, 'log': True},
    'phi_num_leaves':        {'type': 'int',   'low': 45,     'high': 90},
    'phi_max_depth':         {'type': 'int',   'low': 3,      'high': 6},
    'phi_min_child_samples': {'type': 'int',   'low': 130,    'high': 340},
}
TAU_PI_RANGE = (0.84, 1.0)   # τ_π 탐색 범위 (anchor 0.944 주변)

print(f'EXP_ID={EXP_ID} | USER={USER}')
print(f'N_TRIALS={N_TRIALS} | N_FOLDS={N_FOLDS} | N_JOBS={N_JOBS}')
print(f'OUT_DIR={OUT_DIR}')
print(f'DB_PATH={DB_PATH}')
print(f'ZIT search HP={len(ZIT_SEARCH)} + tau_pi={1} = {len(ZIT_SEARCH)+1}')

## 3. 데이터 로드 + 전처리 캐시 준비 (PP는 Optuna 6축 — pp_hpo)


In [ ]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

# train y의 극단값(1.0, 1건)만 두 번째로 큰 값으로 clip — 학습 입력 안정화 (원본 ys는 보존)
ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 -> {second_max:.6f} clip, {n_clipped}개 샘플')

# PP는 더 이상 1회 고정이 아니라 Optuna 6축 — trial마다 호출하는 캐시된 전처리 함수
# (preprocess.run(Stage0->cleaning->winsorize) + add_meta_features(position='raw', die_xy)를 PP 조합별로 캐시)
print('[PP 탐색 6축]')
for _k, _v in pp_hpo.PP_SEARCH_CANDIDATES.items():
    print(f'  {_k:28s} {_v}')
cached_prep = pp_hpo.make_cached_preprocess(
    xs, ys_input, feat_cols, xs_dict,
    position_mode='raw', use_die_xy=True, maxsize=PP_CACHE_SIZE, suppress_stdout=True,
)

# PP와 무관한 것들 (KEY_COL/DIE_KEY_COL/position은 전처리해도 안 바뀜) — 전처리 전 xs_dict에서 한 번만
uid_train_die = xs_dict['train'][KEY_COL].values
uid_val_die   = xs_dict['validation'][KEY_COL].values
uid_test_die  = xs_dict['test'][KEY_COL].values
y_train_unit_s = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_unit_s   = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_unit_s  = ys_input['test'].set_index(KEY_COL)[TARGET_COL]
n_train_die = len(uid_train_die)
n_val_die   = len(uid_val_die)
n_test_die  = len(uid_test_die)
# die-level 학습 target = 각 die에 자기 unit의 health를 broadcast
y_train_die = xs_dict['train'][KEY_COL].map(y_train_unit_s).values.astype(np.float64)

print(f'[데이터] train die={n_train_die:,}, unit train={len(y_train_unit_s):,}, val={len(y_val_unit_s):,}, test={len(y_test_unit_s):,}')
print(f'[cached_preprocess 준비] maxsize={PP_CACHE_SIZE} (X 행렬·feat_cols·xs_*는 PP에 따라 달라 objective/refit 안에서 cached_prep로 생성)')


## 4. K-fold split + Optuna objective

- KFold는 **unit ID 단위 분할** (strategy_common §6) — 같은 unit의 4 die는 같은 fold
- 매 trial: 5 fold 학습 + die-level π/μ → τ_π 적용 → unit 평균 → OOF unit RMSE
- pruning: MedianPruner(n_warmup_steps=2)

In [ ]:
# unit ID 단위 K-fold — 같은 unit의 die 4개는 반드시 같은 fold (leakage 방지). 모든 trial이 공유
unique_units = y_train_unit_s.index.values
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(kf.split(unique_units))   # [(train_idx, val_idx), ...] — unique_units에 대한 인덱스


# die 예측 → unit 예측 (단순 평균)
def _mean_die_to_unit(pred_die, uid_die):
    df = pd.DataFrame({KEY_COL: uid_die, 'pred': pred_die})
    return df.groupby(KEY_COL, sort=False)['pred'].mean().reset_index()

# π가 임계값(τ_π) 초과인 die는 예측을 0으로 강제 (structural zero로 본 것)
def _apply_tau_pi(pred_die, pi_die, tau_pi):
    return np.where(pi_die > tau_pi, 0.0, pred_die)


def objective(trial):
    # Optuna가 trial마다 1번 호출 → 반환값(OOF unit RMSE)을 minimize
    t0 = time.time()
    # PP 6축 샘플 → 이 trial의 전처리 (cached_prep — 같은 PP 조합이면 캐시 hit, 재전처리 없음)
    _pp = pp_hpo.pp_search_space(trial)
    _xs_tr, _, _, _fcols, _ = cached_prep(_pp)
    X_train = _xs_tr[_fcols].values.astype(np.float64)
    trial.set_user_attr('pp_params', dict(_pp))

    # 1) HP 샘플링: ZIT_SEARCH 범위에서 μ/π/φ 트리 HP + zeta + n_em_iters, 그리고 τ_π도 함께 탐색
    params = hpo.sample_from_space(trial, ZIT_SEARCH)
    tau_pi = trial.suggest_float('tau_pi', TAU_PI_RANGE[0], TAU_PI_RANGE[1])

    # 2) 탐색 대상이 아닌 모델 고정 인자 보강 (재현성/환경)
    params['random_state'] = SEED
    params['n_jobs']       = N_JOBS
    params['verbose']      = -1
    params['device']       = 'cpu'      # ZIT은 EM 반복이라 GPU 이득 없음
    params['em_tol']       = 1e-7       # EM 수렴 판정 허용오차

    fold_oof_rmse = []                                                          # fold별 RMSE 누적 (pruning 신호)
    oof_pred_unit = pd.Series(np.nan, index=y_train_unit_s.index, dtype=np.float64)  # 전 train unit OOF 예측 버퍼

    # 3) 5-fold 루프
    for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
        # 3a) fold의 unit ID → die-level 마스크
        tr_units = unique_units[tr_uidx]
        vl_units = unique_units[vl_uidx]
        tr_mask = np.isin(uid_train_die, tr_units)
        vl_mask = np.isin(uid_train_die, vl_units)

        # 3b) ZITboost 학습 (die-level X, die에 broadcast된 unit y)
        model = ZITboostRegressor(**params)
        model.fit(X_train[tr_mask], y_train_die[tr_mask])

        # 3c) val die 예측: (1-π)·μ → 음수 clip → τ_π 적용 → unit 평균
        pi_vl, mu_vl, _ = model.predict_components(X_train[vl_mask])
        pred_die_raw = np.clip((1 - pi_vl) * mu_vl, 0, None)
        pred_die_taupi = _apply_tau_pi(pred_die_raw, pi_vl, tau_pi)
        unit_pred_df = _mean_die_to_unit(pred_die_taupi, uid_train_die[vl_mask])

        # 3d) OOF 버퍼에 기록 + 이 fold의 unit RMSE
        oof_pred_unit.loc[unit_pred_df[KEY_COL].values] = unit_pred_df['pred'].values
        y_vl = y_train_unit_s.loc[unit_pred_df[KEY_COL].values].values
        fold_rmse = float(np.sqrt(np.mean((unit_pred_df['pred'].values - y_vl) ** 2)))
        fold_oof_rmse.append(fold_rmse)

        # 3e) 누적 평균 RMSE를 Optuna에 보고 → MedianPruner가 가망 없으면 조기 종료
        avg = float(np.mean(fold_oof_rmse))
        trial.report(avg, step=fold_idx)
        if trial.should_prune():
            trial.set_user_attr('pruned_at_fold', fold_idx + 1)   # 몇 fold만에 끊겼는지
            trial.set_user_attr('elapsed_sec', time.time() - t0)
            trial.set_user_attr('tau_pi', tau_pi)
            raise optuna.TrialPruned()

    # 4) 5 fold가 train unit 전체를 덮었는지 검증 (남은 NaN = 마스크 버그)
    if oof_pred_unit.isna().any():
        raise RuntimeError('OOF NaN — fold 누락')

    # 5) 전체 OOF unit RMSE = 이 trial의 점수
    oof_rmse = float(np.sqrt(np.mean((oof_pred_unit.values - y_train_unit_s.values) ** 2)))
    elapsed = time.time() - t0
    trial.set_user_attr('elapsed_sec', elapsed)
    trial.set_user_attr('tau_pi', tau_pi)                # refit/후처리에서 best τ_π를 복원해 써야 함
    trial.set_user_attr('fold_oof_rmse', fold_oof_rmse)  # fold별 분산 확인용
    print(f'  trial #{trial.number}: τ_π={tau_pi:.3f}, oof={oof_rmse:.6f}, elapsed={elapsed:.0f}s')
    return oof_rmse


print(f'fold split: {N_FOLDS} folds, unit 단위 분할, seed={SEED}')

## 5. Optuna study 생성 + anchor enqueue + optimize

- TPESampler(seed=None, multivariate=True, group=True) — strategy_common §4
- `enqueue_anchor`로 첫 trial은 1차 best HP 그대로 (strategy_common §5)

In [ ]:
# TPE: multivariate=HP 간 결합 분포 학습, group=조건부 축 자동 skip, seed=None → run마다 다양성
sampler = TPESampler(
    seed=None,
    multivariate=True,
    group=True,
    n_startup_trials=N_STARTUP_TRIALS,
)
# MedianPruner: startup trial 이후, warmup 2 step을 지나(=3번째 fold부터) 같은 시점 다른 trial 중앙값보다 나쁘면 조기 종료
pruner = MedianPruner(n_startup_trials=N_STARTUP_TRIALS, n_warmup_steps=2)

study = optuna.create_study(
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    sampler=sampler,
    pruner=pruner,
    direction='minimize',
    load_if_exists=RESUME,   # RESUME=True면 같은 db에서 이어서, False면 db가 이미 있으면 에러
)

# anchor(=1차 best HP + τ_π)를 study의 trial 0으로 강제 enqueue — 단, 이미 trial이 있으면(RESUME) skip
ANCHOR_FOR_ENQUEUE = dict(ZIT_ONLY_ANCHOR)
ANCHOR_FOR_ENQUEUE['tau_pi'] = ANCHOR_TAU_PI
ANCHOR_FOR_ENQUEUE.update({   # 1차 고정 전처리 값을 pp_* 키로 (corr 0.90은 후보 [0.80,0.84,0.88,0.92,0.96,0.98]에 없어 0.88)
    'pp_missing_threshold': 0.30, 'pp_corr_threshold': 0.88, 'pp_add_indicator': True,
    'pp_indicator_threshold': 0.05, 'pp_spatial_max_dist': 6.0, 'pp_post_impute_corr_threshold': 0.96,
})
if len(study.trials) == 0:
    hpo.enqueue_anchor(study, ANCHOR_FOR_ENQUEUE)
else:
    print(f'[enqueue skip] 기존 trial {len(study.trials)} 있음 — resume')

# 재현성 메타를 study에 박제 (DB에 같이 저장됨)
study_meta = {
    'exp_id': EXP_ID, 'user': USER, 'model': 'ZITboost (zit_only)',
    'n_trials': N_TRIALS, 'n_folds': N_FOLDS, 'n_jobs': N_JOBS,
    'pp_search_candidates': pp_hpo.PP_SEARCH_CANDIDATES, 'anchor': ZIT_ONLY_ANCHOR, 'anchor_tau_pi': ANCHOR_TAU_PI,
    'sampler': 'TPE seed=None multivariate group',
    'pruner':  f'MedianPruner n_startup={N_STARTUP_TRIALS} n_warmup=2',
    'CLIP_Y_EXTREME': CLIP_Y_EXTREME, 'SEED': int(SEED),
}
for k, v in study_meta.items():
    study.set_user_attr(k, str(v))

print(f'study: {study.study_name}, DB: {DB_PATH}')
print(f'기존 trial: {len(study.trials)}')

# HPO 실행 — trial은 직렬(n_jobs=1)로 (모델 내부 N_JOBS와 곱해져 코어가 과할당되는 걸 막음)
t_start = time.time()
study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_SEC, n_jobs=1, show_progress_bar=True)
print(f'\n[HPO 완료] 전체 {time.time()-t_start:.0f}s, total trials={len(study.trials)}')
print(f'  best OOF RMSE: {study.best_value:.6f}')

## 6. Best trial 정보 + anchor enqueue 검증

trial 0이 anchor와 일치하는지 확인 (strategy.md §12 검증 항목).

In [ ]:
best_trial = study.best_trial
best_params = best_trial.params                                            # τ_π 포함 (objective에서 suggest)
best_tau_pi = float(best_trial.user_attrs.get('tau_pi', ANCHOR_TAU_PI))    # user_attr에 따로 기록해 둔 τ_π

print(f'=== Best Trial #{best_trial.number} ===')
print(f'  OOF RMSE  : {best_trial.value:.6f}')
print(f'  best τ_π  : {best_tau_pi:.4f}')
print(f'  elapsed   : {best_trial.user_attrs.get("elapsed_sec", 0):.0f}s')
for k, v in sorted(best_params.items()):
    print(f'    {k}: {v}')

# trial 0이 anchor와 정확히 일치하는지 검증 (enqueue가 제대로 됐는지) — float은 1e-9 허용오차, str은 정확히 일치
trial0 = study.trials[0]
anchor_check = all(
    abs(float(trial0.params.get(k, np.nan)) - float(v)) < 1e-9
    if not isinstance(v, str) else trial0.params.get(k) == v
    for k, v in ANCHOR_FOR_ENQUEUE.items()
)
print(f'\n[검증] trial 0 == anchor? {anchor_check}')

## 7. Best HP 5-fold refit + die-level π/μ 캡처

In [ ]:
# best PP로 전처리 재실행 (best PP 조합은 보통 캐시에 이미 있음) → X 행렬·feat_cols·xs_* 재구성
_best_pp = pp_hpo.pp_params_from_best(best_params)
xs_train, xs_val, xs_test, feat_cols_clean, _eff_pp = cached_prep(_best_pp)
X_train = xs_train[feat_cols_clean].values.astype(np.float64)
X_val   = xs_val[feat_cols_clean].values.astype(np.float64)
X_test  = xs_test[feat_cols_clean].values.astype(np.float64)
print(f"[best PP] {_best_pp}")
print(f"[best PP 전처리 후 feat_cols] {len(feat_cols_clean)} | PP 캐시: {cached_prep.counters}")

# best HP에서 τ_π를 빼고(모델 인자가 아님) 모델 고정 인자를 다시 보강
best_full_params = {k: v for k, v in best_params.items() if k != 'tau_pi' and not str(k).startswith('pp_')}
best_full_params['random_state'] = SEED
best_full_params['n_jobs']       = N_JOBS
best_full_params['verbose']      = -1
best_full_params['device']       = 'cpu'
best_full_params['em_tol']       = 1e-7

n_train_die = len(X_train)
n_val_die   = len(X_val)
n_test_die  = len(X_test)

# train은 OOF (각 die의 검증 fold 예측), val/test는 fold 평균으로 모음 — π/μ/pred 각각
oof_die_pi   = np.full(n_train_die, np.nan)
oof_die_mu   = np.full(n_train_die, np.nan)
oof_die_pred = np.full(n_train_die, np.nan)

val_die_pi   = np.zeros(n_val_die)
val_die_mu   = np.zeros(n_val_die)
val_die_pred = np.zeros(n_val_die)
test_die_pi   = np.zeros(n_test_die)
test_die_mu   = np.zeros(n_test_die)
test_die_pred = np.zeros(n_test_die)

fold_models = []           # fold별 fitted 모델 (pkl 저장용)
em_history_per_fold = []   # fold별 EM 수렴 기록

print(f'=== Best HP 5-fold refit ===')
t0 = time.time()
for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):   # objective와 동일한 FOLDS 사용
    tr_units = unique_units[tr_uidx]
    vl_units = unique_units[vl_uidx]
    tr_mask = np.isin(uid_train_die, tr_units)
    vl_mask = np.isin(uid_train_die, vl_units)

    model = ZITboostRegressor(**best_full_params)
    model.fit(X_train[tr_mask], y_train_die[tr_mask])

    # 이 fold 검증분의 die-level π/μ/pred를 OOF 자리에 채움
    pi_vl, mu_vl, _ = model.predict_components(X_train[vl_mask])
    oof_die_pi[vl_mask]   = pi_vl
    oof_die_mu[vl_mask]   = mu_vl
    oof_die_pred[vl_mask] = np.clip((1 - pi_vl) * mu_vl, 0, None)

    # val/test는 5 fold 모델 예측의 평균
    pi_v, mu_v, _ = model.predict_components(X_val)
    pi_t, mu_t, _ = model.predict_components(X_test)
    val_die_pi    += pi_v / N_FOLDS
    val_die_mu    += mu_v / N_FOLDS
    val_die_pred  += np.clip((1 - pi_v) * mu_v, 0, None) / N_FOLDS
    test_die_pi   += pi_t / N_FOLDS
    test_die_mu   += mu_t / N_FOLDS
    test_die_pred += np.clip((1 - pi_t) * mu_t, 0, None) / N_FOLDS

    fold_models.append(model)
    em_history_per_fold.append(model.em_history_)
    print(f'  fold {fold_idx+1}/{N_FOLDS} done ({time.time()-t0:.0f}s)')

assert not np.isnan(oof_die_pred).any(), 'OOF die pred 미커버'

# EM이 단조 감소(수렴)했는지 점검 — ZITboost는 em_history에 'rmse'(die-level), BagZIT는 'unit_rmse' 키 → 자동 감지
print(f'\n[EM 수렴 체크]')
for f, hist in enumerate(em_history_per_fold):
    key = 'unit_rmse' if hist and 'unit_rmse' in hist[0] else 'rmse'
    rmses = [h[key] for h in hist]
    monotonic = all(rmses[i+1] <= rmses[i] + 1e-6 for i in range(len(rmses)-1))
    print(f'  fold {f+1}: {len(hist)} EM iter, last_{key}={rmses[-1]:.6f}, monotonic_decreasing={monotonic}')

print(f'\n[refit 완료] die-level π/μ/pred 캡처 OK')

## 8. 후처리 — τ_π 적용 → 집계 8 + position Optuna + zero_clip(log)

- 분류 threshold (§9): SKIP (τ_π가 die-level 역할)
- 집계 다양성 (§10): 8종 (Q25/Q75 추가)
- Position 가중치 (§11): Optuna sub-study 50 trial
- zero_clip (§12): log space 비교

In [ ]:
# best τ_π를 die-level 예측에 적용 (π가 큰 die는 0으로)
oof_die_pred_taupi  = _apply_tau_pi(oof_die_pred,  oof_die_pi,  best_tau_pi)
val_die_pred_taupi  = _apply_tau_pi(val_die_pred,  val_die_pi,  best_tau_pi)
test_die_pred_taupi = _apply_tau_pi(test_die_pred, test_die_pi, best_tau_pi)

# τ_π로 0 처리된 die 비율 (참고)
killed = {
    'oof':  float((oof_die_pi  > best_tau_pi).mean()),
    'val':  float((val_die_pi  > best_tau_pi).mean()),
    'test': float((test_die_pi > best_tau_pi).mean()),
}
print(f'[τ_π={best_tau_pi:.3f} 적용] 0 처리 die 비율: {killed}')

# 후처리: die→unit 집계(8종 중 best) + position 가중평균(Optuna 50t) + zero_clip(log 공간) — 각 단계 val 개선 시만 채택.
# π threshold는 끔(use_pi_threshold=False): 이미 τ_π로 die-level에서 처리했으므로 중복 적용 안 함.
pp_res = postprocess.tune_and_apply(
    xs_train, xs_val, xs_test,
    die_pred_train=oof_die_pred_taupi,
    die_pred_val=val_die_pred_taupi,
    die_pred_test=test_die_pred_taupi,
    y_train_unit=ys_input['train'],
    y_val_unit=ys_input['validation'],     # val로 채택 여부 결정
    use_pi_threshold=False,                # τ_π가 die-level π 게이트 역할을 이미 함
    agg_methods=postprocess.AGG_METHODS,   # mean/median/max/min/trimmed_mean/weighted/Q25/Q75
    zero_clip_log_space=True,              # log1p 공간에서 zero_clip 임계값 비교
    position_method='optuna',
    position_optuna_n_trials=50,
)

print(f'\n[Postprocess]')
print(f'  best_agg            : {pp_res["best_agg"]}')
print(f'  pos_weights         : {pp_res["pos_weights"]}')
print(f'  best_zero_clip(log) : {pp_res["best_zero_clip"]}')
print(f'  position_method     : {pp_res["position_method"]}')
print(f'  train_rmse          : {pp_res["train_rmse"]:.6f}')
print(f'  val_rmse_final      : {pp_res.get("val_rmse_final")}')

# 후처리 적용본의 val/test unit RMSE 직접 계산 (정답과 정렬)
if pp_res.get('final_val_unit') is not None:
    _val_pred = pp_res['final_val_unit'].set_index(KEY_COL)['pred'].loc[y_val_unit_s.index]
    val_rmse  = float(np.sqrt(np.mean((_val_pred.values  - y_val_unit_s.values)  ** 2)))
    print(f'  val_rmse            : {val_rmse:.6f}')
if pp_res.get('final_test_unit') is not None:
    _test_pred = pp_res['final_test_unit'].set_index(KEY_COL)['pred'].loc[y_test_unit_s.index]
    test_rmse  = float(np.sqrt(np.mean((_test_pred.values - y_test_unit_s.values) ** 2)))
    print(f'  test_rmse           : {test_rmse:.6f}')

print(f'  agg_rmses           : {pp_res["agg_rmses"]}')
print(f'  decisions           :')
for k, v in pp_res.get('decisions', {}).items():
    print(f'    {k:14s} {v}')

## 9. 산출물 9개 저장 (strategy_common §15)

best_params.json + fold_models.pkl + 6 CSV (die ×3 + unit ×3) + optuna_*.db

In [ ]:
import json, pickle, hashlib

# 1) fold_models.pkl — fold별 모델 + feature 이름 + EM 기록 (추론 시 그대로 로드)
with open(os.path.join(OUT_DIR, 'fold_models.pkl'), 'wb') as f:
    pickle.dump({
        'fold_models':         fold_models,
        'feature_names':       feat_cols_clean,
        'model_name':          'zitboost',
        'n_folds':             N_FOLDS,
        'em_history_per_fold': em_history_per_fold,
    }, f)

# 2) best_params.json — 재현성 메타. train unit 목록의 해시도 박제 (다른 단계 OOF와 같은 분할인지 검증용)
uid_arr = ys_input['train'][KEY_COL].unique()
unit_ids_hash = hashlib.sha1(','.join(map(str, uid_arr)).encode()).hexdigest()

best_meta = {
    'exp_id':                EXP_ID,
    'model_name':            'zitboost',
    'best_trial_number':     best_trial.number,
    'best_oof_rmse':         float(best_trial.value),
    'best_params_resolved':  best_full_params,
    'best_tau_pi':           best_tau_pi,
    'feature_names':         feat_cols_clean,
    'n_features':            len(feat_cols_clean),
    'n_folds':               N_FOLDS,
    'unit_ids_hash':         unit_ids_hash,
    'n_units_train':         int(len(uid_arr)),
    'effective_pp_params':   _eff_pp,
    'best_pp_params':        _best_pp,
    'study_meta':            study_meta,
    'postprocess': {
        'best_agg':            pp_res['best_agg'],
        'pos_weights':         pp_res['pos_weights'].tolist() if pp_res['pos_weights'] is not None else None,
        'best_zero_clip':      float(pp_res['best_zero_clip']) if pp_res['best_zero_clip'] is not None else None,
        'zero_clip_log_space': pp_res['zero_clip_log_space'],
        'position_method':     pp_res['position_method'],
        'agg_rmses':           {k: float(v) for k, v in pp_res['agg_rmses'].items()},
        'train_rmse':          float(pp_res['train_rmse']),
    },
}
with open(os.path.join(OUT_DIR, 'best_params.json'), 'w', encoding='utf-8') as f:
    json.dump(best_meta, f, indent=2, ensure_ascii=False, default=str)

# 3-5) die-level 예측 CSV — π/(1-π)/μ/pred + health. (1-π)는 다른 노트북에서 multiplier로 쓰기 쉽게 미리 만듦
def _build_die_df(uid, die_id, position, pi, mu, pred, y_unit):
    df = pd.DataFrame({
        KEY_COL: uid, DIE_KEY_COL: die_id, 'position': position,
        'pi': pi, 'one_minus_pi': 1.0 - pi, 'mu': mu, 'pred': pred,
    })
    if y_unit is not None:
        df[TARGET_COL] = df[KEY_COL].map(y_unit)   # unit health를 die에 broadcast
    return df

_build_die_df(
    uid_train_die, xs_train[DIE_KEY_COL].values, xs_train['position'].values,
    oof_die_pi, oof_die_mu, oof_die_pred, y_train_unit_s,
).to_csv(os.path.join(OUT_DIR, 'oof_die.csv'), index=False)
_build_die_df(
    uid_val_die, xs_val[DIE_KEY_COL].values, xs_val['position'].values,
    val_die_pi, val_die_mu, val_die_pred, y_val_unit_s,
).to_csv(os.path.join(OUT_DIR, 'val_die.csv'), index=False)
_build_die_df(
    uid_test_die, xs_test[DIE_KEY_COL].values, xs_test['position'].values,
    test_die_pi, test_die_mu, test_die_pred, y_test_unit_s,
).to_csv(os.path.join(OUT_DIR, 'test_die.csv'), index=False)

# 6-8) unit-level 예측 CSV — 후처리 적용본 + health
def _build_unit_df(unit_pred_df, y_unit):
    out = unit_pred_df.copy()
    out[TARGET_COL] = out[KEY_COL].map(y_unit)
    return out

_build_unit_df(pp_res['final_train_unit'], y_train_unit_s).to_csv(os.path.join(OUT_DIR, 'oof_unit.csv'),  index=False)
_build_unit_df(pp_res['final_val_unit'],   y_val_unit_s  ).to_csv(os.path.join(OUT_DIR, 'val_unit.csv'),  index=False)
_build_unit_df(pp_res['final_test_unit'],  y_test_unit_s ).to_csv(os.path.join(OUT_DIR, 'test_unit.csv'), index=False)

# 9) optuna_*.db — study.optimize가 학습 중 자동 저장 (별도 코드 불필요)

# 저장된 파일 목록 출력
print(f'\n저장 완료: {OUT_DIR}')
for fn in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, fn)) / 1024
    print(f'  {fn:30s}  {sz:10,.1f} KB')

# Colab이면 산출물 폴더를 zip으로 묶어 로컬 PC로 다운로드
try:
    import google.colab
    from google.colab import files
    import shutil
    _zip = shutil.make_archive(os.path.join('/content', f'zit_only_{EXP_ID}_outputs'), 'zip', OUT_DIR)
    print(f'\n[zip 생성] {_zip} ({os.path.getsize(_zip)/1024:.1f} KB)')
    try:
        files.download(_zip)
    except Exception as _e:
        from IPython.display import FileLink, display
        print(f'[files.download 실패: {_e}] 아래 링크 클릭')
        display(FileLink(_zip))
except ImportError:
    pass